# ✅ THEORY 
#### Missing Indicator = an extra feature (column) that tells:
####   1  -> value was missing (NaN)
####   0  -> value was present
####
#### Why useful?
#### Sometimes "missingness" itself is an important pattern for prediction.
#### Example: If Age is missing for many passengers, that can affect survival.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# Step 1) Load dataset
df = pd.read_csv('train.csv', usecols=['Age', 'Fare', 'Survived'])
print(df.head(), "\n")
print("Missing %:\n", df.isnull().mean() * 100, "\n")

   Survived   Age     Fare
0         0  22.0   7.2500
1         1  38.0  71.2833
2         1  26.0   7.9250
3         1  35.0  53.1000
4         0  35.0   8.0500 

Missing %:
 Survived     0.00000
Age         19.86532
Fare         0.00000
dtype: float64 



In [3]:
# Step 2) Split into features (X) and target (y)
X = df.drop(columns=['Survived'])
y = df['Survived']


In [4]:
# Step 3) Train-test split (always split before preprocessing to avoid data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# ✅ Make safe copies (prevents SettingWithCopyWarning)
X_train = X_train.copy()
X_test = X_test.copy()

print("Train missing Age:", X_train['Age'].isnull().sum())
print("Test missing Age:", X_test['Age'].isnull().sum(), "\n")


Train missing Age: 148
Test missing Age: 29 



## ✅ BASELINE: Only SimpleImputer (no Missing Indicator)
#### SimpleImputer() will fill missing numerical values using mean by default.

In [5]:
si = SimpleImputer()

X_train_imputed = si.fit_transform(X_train)  # fit only on train
X_test_imputed = si.transform(X_test)        # apply same rule on test

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_imputed, y_train)

y_pred = clf.predict(X_test_imputed)
print("Accuracy WITHOUT Missing Indicator:", accuracy_score(y_test, y_pred))



Accuracy WITHOUT Missing Indicator: 0.6145251396648045


## ✅ MANUAL Missing Indicator
### MissingIndicator finds which columns have missing values
### and creates a True/False column for missing values.

In [6]:
mi = MissingIndicator()
mi.fit(X_train)

print("\nMissingIndicator column index:", mi.features_)
# Example: [0] means column 0 (Age) has missing values

X_train_missing = mi.transform(X_train)  # True/False array
X_test_missing = mi.transform(X_test)

# Add indicator column (convert True/False to 1/0)
X_train["Age_NA"] = X_train_missing.astype(int)
X_test["Age_NA"] = X_test_missing.astype(int)

print("\nX_train with Age_NA column:")
print(X_train.head(), "\n")


MissingIndicator column index: [0]

X_train with Age_NA column:
      Age     Fare  Age_NA
30   40.0  27.7208       0
10    4.0  16.7000       0
873  47.0   9.0000       0
182   9.0  31.3875       0
876  20.0   9.8458       0 



In [7]:
# ==========================================================
# ✅ Now we impute again + train model again
# Model input = [Age (imputed), Fare, Age_NA]
# ==========================================================
si2 = SimpleImputer()

X_train_final = si2.fit_transform(X_train)
X_test_final = si2.transform(X_test)

clf2 = LogisticRegression(max_iter=1000)
clf2.fit(X_train_final, y_train)

y_pred2 = clf2.predict(X_test_final)
print("Accuracy WITH Missing Indicator:", accuracy_score(y_test, y_pred2))


Accuracy WITH Missing Indicator: 0.6312849162011173


## ✅ SHORTCUT: SimpleImputer(add_indicator=True)
#### It automatically:
#### 1) fills missing values
#### 2) adds missing indicator column(s)

In [8]:
si3 = SimpleImputer(add_indicator=True)

X_train_auto = si3.fit_transform(X_train[['Age', 'Fare']])
X_test_auto = si3.transform(X_test[['Age', 'Fare']])

clf3 = LogisticRegression(max_iter=1000)
clf3.fit(X_train_auto, y_train)

y_pred3 = clf3.predict(X_test_auto)
print("Accuracy with SimpleImputer(add_indicator=True):", accuracy_score(y_test, y_pred3))

Accuracy with SimpleImputer(add_indicator=True): 0.6312849162011173
